In [ ]:
"""
Enhanced Grazioso Salvare Rescue Candidate Dashboard.

This dashboard uses the reusable data preparation, intake-to-outcome
matching, and rescue suitability scoring modules created for the
CS 499 enhancement.
"""

from __future__ import annotations

import base64
import os
import re
import threading
import time
from typing import Any

import dash_leaflet as dl
import requests
from dash import (
    Dash,
    Input,
    Output,
    State,
    dash_table,
    dcc,
    html,
    no_update,
)

from dashboard_helpers import (
    TABLE_COLUMN_DEFINITIONS,
    build_selected_row_styles,
    get_selected_candidate,
    get_table_dataframe,
    prepare_dashboard_results,
    prepare_export_dataframe,
)
from run_full_matching import build_ranked_candidate_data


#############################
# Data Manipulation / Model
#############################

# Run the full pipeline once when the dashboard starts. Callbacks reuse
# these results instead of rebuilding more than 150,000 matches.
matched_records, ranked_candidates = build_ranked_candidate_data()

# Prepare the three rescue mission views and the combined All view.
dashboard_results = prepare_dashboard_results(
    ranked_candidates
)

# Start with all candidates and the strongest mission for each animal.
initial_dataframe = get_table_dataframe(
    dashboard_results,
    "all",
)


#########################
# Dashboard Configuration
#########################

app = Dash(__name__)

AUSTIN_CENTER = [
    30.2672,
    -97.7431,
]

NO_LOCATION_MESSAGE = (
    "No location information"
)

PAGE_STYLE = {
    "minHeight": "100vh",
    "backgroundColor": "#f3f5f7",
    "fontFamily": (
        "system-ui, -apple-system, BlinkMacSystemFont, "
        "Segoe UI, Roboto, sans-serif"
    ),
    "color": "#1f2937",
}

CONTENT_STYLE = {
    "maxWidth": "1500px",
    "margin": "0 auto",
    "padding": "22px",
}

CARD_STYLE = {
    "backgroundColor": "white",
    "border": "1px solid #dfe3e8",
    "borderRadius": "12px",
    "boxShadow": "0 3px 12px rgba(0, 0, 0, 0.07)",
    "padding": "18px",
}

SECTION_TITLE_STYLE = {
    "margin": "0 0 16px 0",
    "fontSize": "22px",
    "color": "#263238",
}

BUTTON_STYLE = {
    "height": "42px",
    "padding": "0 18px",
    "border": "1px solid #9ca3af",
    "borderRadius": "7px",
    "backgroundColor": "white",
    "cursor": "pointer",
    "fontSize": "14px",
    "fontWeight": "600",
}

MISSION_LABEL_STYLE = {
    "display": "block",
    "padding": "11px 14px",
    "marginBottom": "10px",
    "border": "1px solid #c7cdd4",
    "borderRadius": "8px",
    "backgroundColor": "#f8fafc",
    "cursor": "pointer",
    "fontSize": "15px",
}

DETAIL_LABEL_STYLE = {
    "fontSize": "12px",
    "fontWeight": "700",
    "textTransform": "uppercase",
    "letterSpacing": "0.04em",
    "color": "#6b7280",
    "marginBottom": "3px",
}

DETAIL_VALUE_STYLE = {
    "fontSize": "15px",
    "fontWeight": "600",
    "color": "#1f2937",
    "overflowWrap": "anywhere",
}


#########################
# Logo
#########################

image_filename = (
    "Grazioso Salvare Logo.png"
)

if os.path.exists(image_filename):
    with open(
        image_filename,
        "rb",
    ) as image_file:
        encoded_image = base64.b64encode(
            image_file.read()
        ).decode("utf-8")
else:
    print(
        "Logo file not found:",
        image_filename,
    )
    encoded_image = None


#########################
# Location Helpers
#########################

# Successful results are cached for the current dashboard session.
# Failed searches are not cached because a temporary network issue
# should not cause an address to remain unavailable.
GEOCODE_CACHE: dict[
    str,
    tuple[float, float],
] = {}

GEOCODE_LOCK = threading.Lock()
LAST_GEOCODE_REQUEST_TIME = 0.0

GEOCODE_MINIMUM_INTERVAL = 1.05
GEOCODE_TIMEOUT = 8
GEOCODE_RETRY_COUNT = 2

GEOCODE_URL = (
    "https://nominatim.openstreetmap.org/search"
)

GEOCODE_HEADERS = {
    "User-Agent": (
        "Grazioso-Salvare-CS499-Dashboard/1.0 "
        "(educational capstone project)"
    ),
    "Accept-Language": "en-US,en;q=0.9",
}

# This view box gives the geocoder a preference for the Austin and
# Travis County area without forcing every result to remain in Austin.
CENTRAL_TEXAS_VIEWBOX = (
    "-98.35,30.75,-96.95,29.75"
)


def has_usable_location(
    location: Any,
) -> bool:
    """
    Return True when a location contains usable text.
    """

    if location is None:
        return False

    clean_location = str(
        location
    ).strip()

    return clean_location not in {
        "",
        "<NA>",
        "nan",
        "None",
        "Not available",
        NO_LOCATION_MESSAGE,
    }


def normalize_location(
    location: Any,
) -> str:
    """
    Convert shelter location text into a cleaner geocoding query.
    """

    if not has_usable_location(
        location
    ):
        return ""

    clean_location = str(
        location
    ).strip()

    # Convert:
    # "4601 Imperial Dr in Austin (TX)"
    # into:
    # "4601 Imperial Dr, Austin, TX"
    clean_location = re.sub(
        r"\s+in\s+",
        ", ",
        clean_location,
        flags=re.IGNORECASE,
    )

    clean_location = clean_location.replace(
        "(TX)",
        "TX",
    )

    clean_location = clean_location.replace(
        "&",
        " and ",
    )

    clean_location = re.sub(
        r"\s*,\s*",
        ", ",
        clean_location,
    )

    clean_location = re.sub(
        r"\s+",
        " ",
        clean_location,
    ).strip(" ,")

    if not re.search(
        r"\bTX\b|\bTexas\b",
        clean_location,
        flags=re.IGNORECASE,
    ):
        clean_location = (
            f"{clean_location}, Texas"
        )

    return clean_location


def expand_street_abbreviations(
    location: str,
) -> str:
    """
    Expand common street abbreviations for a fallback search.
    """

    replacements = {
        r"\bRd\b": "Road",
        r"\bDr\b": "Drive",
        r"\bAve\b": "Avenue",
        r"\bSt\b": "Street",
        r"\bLn\b": "Lane",
        r"\bBlvd\b": "Boulevard",
        r"\bCir\b": "Circle",
        r"\bCt\b": "Court",
        r"\bHwy\b": "Highway",
        r"\bFm\b": "Farm to Market Road",
        r"\bPkwy\b": "Parkway",
        r"\bPl\b": "Place",
        r"\bTrl\b": "Trail",
        r"\bCv\b": "Cove",
        r"\bTer\b": "Terrace",
    }

    expanded_location = location

    for pattern, replacement in replacements.items():
        expanded_location = re.sub(
            pattern,
            replacement,
            expanded_location,
            flags=re.IGNORECASE,
        )

    return expanded_location


def build_geocode_queries(
    location: Any,
) -> list[str]:
    """
    Build multiple reasonable search queries for one location.
    """

    normalized_location = normalize_location(
        location
    )

    if not normalized_location:
        return []

    expanded_location = expand_street_abbreviations(
        normalized_location
    )

    queries = [
        normalized_location,
    ]

    if (
        expanded_location.casefold()
        != normalized_location.casefold()
    ):
        queries.append(
            expanded_location
        )

    # Add Austin or Travis County when the record contains a street
    # but does not include a recognizable city or county.
    contains_street_number = bool(
        re.search(
            r"\b\d+\b",
            normalized_location,
        )
    )

    contains_known_area = bool(
        re.search(
            (
                r"\bAustin\b|\bTravis\b|\bDel Valle\b|"
                r"\bManor\b|\bPflugerville\b|\bLeander\b|"
                r"\bRound Rock\b|\bCedar Park\b|\bBuda\b|"
                r"\bKyle\b|\bBee Cave\b|\bLakeway\b|"
                r"\bBastrop\b"
            ),
            normalized_location,
            flags=re.IGNORECASE,
        )
    )

    if (
        contains_street_number
        and not contains_known_area
    ):
        queries.append(
            f"{normalized_location}, Austin"
        )
        queries.append(
            f"{expanded_location}, Travis County"
        )

    # Keep only unique queries while preserving their order.
    unique_queries = []

    for query in queries:
        cleaned_query = re.sub(
            r"\s+",
            " ",
            query,
        ).strip(" ,")

        if (
            cleaned_query
            and cleaned_query not in unique_queries
        ):
            unique_queries.append(
                cleaned_query
            )

    return unique_queries


def wait_for_geocode_interval() -> None:
    """
    Enforce the public geocoder's request interval.
    """

    global LAST_GEOCODE_REQUEST_TIME

    elapsed_time = (
        time.monotonic()
        - LAST_GEOCODE_REQUEST_TIME
    )

    remaining_delay = (
        GEOCODE_MINIMUM_INTERVAL
        - elapsed_time
    )

    if remaining_delay > 0:
        time.sleep(
            remaining_delay
        )

    LAST_GEOCODE_REQUEST_TIME = (
        time.monotonic()
    )


def request_geocode(
    query: str,
) -> tuple[float, float] | None:
    """
    Send one rate-limited request to the geocoding service.
    """

    for attempt in range(
        GEOCODE_RETRY_COUNT
    ):
        with GEOCODE_LOCK:
            wait_for_geocode_interval()

            try:
                response = requests.get(
                    GEOCODE_URL,
                    params={
                        "q": query,
                        "format": "jsonv2",
                        "limit": 1,
                        "countrycodes": "us",
                        "addressdetails": 1,
                        "viewbox": (
                            CENTRAL_TEXAS_VIEWBOX
                        ),
                        "bounded": 0,
                    },
                    headers=GEOCODE_HEADERS,
                    timeout=GEOCODE_TIMEOUT,
                )

                # Retry temporary service failures and rate limits.
                if response.status_code in {
                    429,
                    500,
                    502,
                    503,
                    504,
                }:
                    if (
                        attempt
                        < GEOCODE_RETRY_COUNT - 1
                    ):
                        continue

                    return None

                response.raise_for_status()

                results = response.json()

                if not results:
                    return None

                latitude = float(
                    results[0]["lat"]
                )
                longitude = float(
                    results[0]["lon"]
                )

                return (
                    latitude,
                    longitude,
                )

            except (
                requests.RequestException,
                KeyError,
                TypeError,
                ValueError,
            ):
                if (
                    attempt
                    >= GEOCODE_RETRY_COUNT - 1
                ):
                    return None

    return None


def geocode_location(
    location: Any,
) -> tuple[float, float] | None:
    """
    Find approximate coordinates using cached and fallback searches.
    """

    normalized_location = normalize_location(
        location
    )

    if not normalized_location:
        return None

    cache_key = (
        normalized_location.casefold()
    )

    cached_coordinates = (
        GEOCODE_CACHE.get(
            cache_key
        )
    )

    if cached_coordinates is not None:
        return cached_coordinates

    queries = build_geocode_queries(
        location
    )

    for query in queries:
        coordinates = request_geocode(
            query
        )

        if coordinates is not None:
            # Only successful searches are cached.
            GEOCODE_CACHE[
                cache_key
            ] = coordinates

            return coordinates

    # Do not cache failed results. The next click can try again in
    # case the service was temporarily unavailable.
    return None


def is_specific_address(
    location: Any,
) -> bool:
    """
    Determine whether a location appears to contain a street address.
    """

    if not has_usable_location(
        location
    ):
        return False

    location_text = str(
        location
    )

    return bool(
        re.search(
            r"\b\d+\b",
            location_text,
        )
    )


def build_map(
    location: Any,
) -> html.Div:
    """
    Create the map for the selected candidate.
    """

    coordinates = geocode_location(
        location
    )

    if coordinates is None:
        location_text = (
            NO_LOCATION_MESSAGE
        )

        map_center = AUSTIN_CENTER
        map_zoom = 10
        marker_children = []

    else:
        latitude, longitude = (
            coordinates
        )

        location_text = str(
            location
        )

        map_center = [
            latitude,
            longitude,
        ]

        map_zoom = (
            15
            if is_specific_address(location)
            else 11
        )

        marker_children = [
            dl.Marker(
                position=map_center,
                children=[
                    dl.Tooltip(
                        str(location)
                    ),
                    dl.Popup(
                        html.Div(
                            [
                                html.Strong(
                                    "Approximate location"
                                ),
                                html.Br(),
                                html.Span(
                                    str(location)
                                ),
                            ]
                        )
                    ),
                ],
            )
        ]

    return html.Div(
        children=[
            html.Div(
                location_text,
                style={
                    "fontWeight": "600",
                    "marginBottom": "10px",
                    "minHeight": "22px",
                },
            ),
            dl.Map(
                center=map_center,
                zoom=map_zoom,
                style={
                    "width": "100%",
                    "height": "320px",
                    "borderRadius": "8px",
                },
                children=[
                    dl.TileLayer(),
                    *marker_children,
                ],
            ),
        ]
    )


def detail_item(
    label: str,
    value: Any,
) -> html.Div:
    """
    Create one labeled value in the selected candidate panel.
    """

    display_value = value

    if (
        display_value is None
        or str(display_value).strip()
        in {
            "",
            "<NA>",
            "nan",
            "None",
        }
    ):
        display_value = (
            "Not available"
        )

    return html.Div(
        children=[
            html.Div(
                label,
                style=DETAIL_LABEL_STYLE,
            ),
            html.Div(
                str(display_value),
                style=DETAIL_VALUE_STYLE,
            ),
        ]
    )


def build_candidate_details(
    candidate: dict[str, Any] | None,
) -> html.Div:
    """
    Create the detailed view for the selected candidate.
    """

    if candidate is None:
        return html.Div(
            "No candidate is currently selected.",
            style={
                "color": "#6b7280",
            },
        )

    location = candidate.get(
        "found_location",
        NO_LOCATION_MESSAGE,
    )

    if not has_usable_location(
        location
    ):
        location = NO_LOCATION_MESSAGE

    return html.Div(
        children=[
            html.Div(
                style={
                    "display": "grid",
                    "gridTemplateColumns": (
                        "repeat(2, minmax(0, 1fr))"
                    ),
                    "gap": "16px",
                },
                children=[
                    detail_item(
                        "Animal ID",
                        candidate.get(
                            "animal_id"
                        ),
                    ),
                    detail_item(
                        "Name",
                        candidate.get(
                            "outcome_name"
                        ),
                    ),
                    detail_item(
                        "Breed",
                        candidate.get(
                            "outcome_breed"
                        ),
                    ),
                    detail_item(
                        "Age",
                        candidate.get(
                            "outcome_age_upon_outcome"
                        ),
                    ),
                    detail_item(
                        "Rescue Mission",
                        candidate.get(
                            "rescue_mission"
                        ),
                    ),
                    detail_item(
                        "Suitability Score",
                        candidate.get(
                            "total_score"
                        ),
                    ),
                    detail_item(
                        "Sex and Intact Status",
                        candidate.get(
                            "outcome_sex_upon_outcome"
                        ),
                    ),
                    detail_item(
                        "Found Location",
                        location,
                    ),
                ],
            ),

            html.Hr(
                style={
                    "margin": "18px 0",
                    "border": "0",
                    "borderTop": (
                        "1px solid #e5e7eb"
                    ),
                }
            ),

            html.Div(
                style={
                    "display": "grid",
                    "gridTemplateColumns": (
                        "repeat(3, minmax(0, 1fr))"
                    ),
                    "gap": "12px",
                    "marginBottom": "18px",
                },
                children=[
                    detail_item(
                        "Breed Score",
                        candidate.get(
                            "breed_score"
                        ),
                    ),
                    detail_item(
                        "Age Score",
                        candidate.get(
                            "age_score"
                        ),
                    ),
                    detail_item(
                        "Sex Score",
                        candidate.get(
                            "sex_score"
                        ),
                    ),
                ],
            ),

            html.Div(
                "Scoring Explanation",
                style=DETAIL_LABEL_STYLE,
            ),
            html.P(
                candidate.get(
                    "score_explanation",
                    (
                        "No scoring explanation "
                        "is available."
                    ),
                ),
                style={
                    "lineHeight": "1.55",
                    "marginBottom": "0",
                },
            ),
        ]
    )


#########################
# Dashboard Layout
#########################

app.layout = html.Div(
    style=PAGE_STYLE,
    children=[
        html.Div(
            style=CONTENT_STYLE,
            children=[
                html.Div(
                    style=CARD_STYLE
                    | {
                        "display": "grid",
                        "gridTemplateColumns": (
                            "180px minmax(0, 1fr)"
                        ),
                        "alignItems": "center",
                        "gap": "26px",
                        "marginBottom": "18px",
                    },
                    children=[
                        html.Div(
                            style={
                                "textAlign": "center",
                            },
                            children=[
                                html.Img(
                                    src=(
                                        "data:image/png;base64,"
                                        f"{encoded_image}"
                                        if encoded_image
                                        else ""
                                    ),
                                    style={
                                        "height": "150px",
                                        "maxWidth": "100%",
                                        "objectFit": "contain",
                                    },
                                )
                            ],
                        ),
                        html.Div(
                            style={
                                "textAlign": "center",
                            },
                            children=[
                                html.H1(
                                    (
                                        "Grazioso Salvare Rescue "
                                        "Candidate Dashboard"
                                    ),
                                    style={
                                        "margin": "0",
                                        "fontSize": "34px",
                                        "color": "#263238",
                                    },
                                ),
                                html.Div(
                                    (
                                        "Dustin Davis | "
                                        "CS 499 Enhanced Artifact"
                                    ),
                                    style={
                                        "marginTop": "10px",
                                        "fontSize": "18px",
                                        "color": "#4b5563",
                                    },
                                ),
                            ],
                        ),
                    ],
                ),

                html.Div(
                    style={
                        "display": "grid",
                        "gridTemplateColumns": (
                            "minmax(300px, 0.75fr) "
                            "minmax(500px, 1.25fr)"
                        ),
                        "gap": "18px",
                        "marginBottom": "18px",
                        "alignItems": "stretch",
                    },
                    children=[
                        html.Div(
                            style=CARD_STYLE,
                            children=[
                                html.H2(
                                    "Rescue Mission Type",
                                    style=SECTION_TITLE_STYLE,
                                ),
                                dcc.RadioItems(
                                    id="filter-type",
                                    options=[
                                        {
                                            "label": (
                                                "All Candidates"
                                            ),
                                            "value": "all",
                                        },
                                        {
                                            "label": (
                                                "Water Rescue"
                                            ),
                                            "value": "water",
                                        },
                                        {
                                            "label": (
                                                "Mountain or "
                                                "Wilderness"
                                            ),
                                            "value": "mountain",
                                        },
                                        {
                                            "label": (
                                                "Disaster or "
                                                "Individual Tracking"
                                            ),
                                            "value": "disaster",
                                        },
                                    ],
                                    value="all",
                                    labelStyle=(
                                        MISSION_LABEL_STYLE
                                    ),
                                    inputStyle={
                                        "marginRight": "9px",
                                        "cursor": "pointer",
                                    },
                                ),
                                html.Div(
                                    style={
                                        "display": "flex",
                                        "gap": "10px",
                                        "flexWrap": "wrap",
                                        "marginTop": "18px",
                                    },
                                    children=[
                                        html.Button(
                                            "Reset Filters",
                                            id="btn-reset",
                                            n_clicks=0,
                                            style=BUTTON_STYLE,
                                        ),
                                        html.Button(
                                            "Export Current View",
                                            id="btn-export",
                                            n_clicks=0,
                                            style=BUTTON_STYLE,
                                        ),
                                        dcc.Download(
                                            id="download-csv"
                                        ),
                                    ],
                                ),
                            ],
                        ),

                        html.Div(
                            style=CARD_STYLE,
                            children=[
                                html.H2(
                                    "Matched Intake Location",
                                    style=SECTION_TITLE_STYLE,
                                ),
                                html.Div(
                                    id="map-id",
                                    children=build_map(
                                        initial_dataframe.iloc[0][
                                            "found_location"
                                        ]
                                        if not initial_dataframe.empty
                                        else NO_LOCATION_MESSAGE
                                    ),
                                ),
                            ],
                        ),
                    ],
                ),

                html.Div(
                    style={
                        "display": "grid",
                        "gridTemplateColumns": (
                            "minmax(700px, 1.35fr) "
                            "minmax(380px, 0.65fr)"
                        ),
                        "gap": "18px",
                        "alignItems": "start",
                    },
                    children=[
                        html.Div(
                            style=CARD_STYLE,
                            children=[
                                html.H2(
                                    "Ranked Rescue Candidates",
                                    style=SECTION_TITLE_STYLE,
                                ),
                                dash_table.DataTable(
                                    id="datatable-id",
                                    css=[
                                        {
                                            "selector": (
                                                ".dash-cell.cell--selected"
                                            ),
                                            "rule": (
                                                "box-shadow: none !important; "
                                                "outline: none !important;"
                                            ),
                                        },
                                        {
                                            "selector": (
                                                ".dash-cell.focused"
                                            ),
                                            "rule": (
                                                "box-shadow: none !important; "
                                                "outline: none !important;"
                                            ),
                                        },
                                    ],
                                    columns=(
                                        TABLE_COLUMN_DEFINITIONS
                                    ),
                                    data=(
                                        initial_dataframe.to_dict(
                                            "records"
                                        )
                                    ),
                                    page_current=0,
                                    page_size=10,
                                    filter_action="native",
                                    sort_action="native",
                                    sort_mode="multi",
                                    active_cell={
                                        "row": 0,
                                        "column": 0,
                                        "column_id": (
                                            "rescue_rank"
                                        ),
                                    },
                                    selected_cells=[],
                                    cell_selectable=True,
                                    style_table={
                                        "overflowX": "auto",
                                        "width": "100%",
                                    },
                                    style_cell={
                                        "textAlign": "left",
                                        "padding": "10px",
                                        "whiteSpace": "normal",
                                        "height": "auto",
                                        "minWidth": "85px",
                                        "maxWidth": "220px",
                                        "border": (
                                            "1px solid #e2e8f0"
                                        ),
                                        "cursor": "pointer",
                                    },
                                    style_cell_conditional=[
                                        {
                                            "if": {
                                                "column_id": (
                                                    "rescue_rank"
                                                )
                                            },
                                            "width": "65px",
                                            "textAlign": "center",
                                        },
                                        {
                                            "if": {
                                                "column_id": (
                                                    "animal_id"
                                                )
                                            },
                                            "width": "105px",
                                        },
                                        {
                                            "if": {
                                                "column_id": (
                                                    "rescue_mission"
                                                )
                                            },
                                            "width": "170px",
                                        },
                                        {
                                            "if": {
                                                "column_id": (
                                                    "outcome_name"
                                                )
                                            },
                                            "width": "130px",
                                        },
                                        {
                                            "if": {
                                                "column_id": (
                                                    "outcome_breed"
                                                )
                                            },
                                            "width": "180px",
                                        },
                                        {
                                            "if": {
                                                "column_id": (
                                                    "outcome_age_upon_outcome"
                                                )
                                            },
                                            "width": "90px",
                                        },
                                        {
                                            "if": {
                                                "column_id": (
                                                    "found_location"
                                                )
                                            },
                                            "width": "240px",
                                        },
                                    ],
                                    style_header={
                                        "fontWeight": "700",
                                        "backgroundColor": "#e8edf2",
                                        "color": "#263238",
                                        "border": (
                                            "1px solid #cfd6dc"
                                        ),
                                    },
                                    style_data_conditional=(
                                        build_selected_row_styles(
                                            {
                                                "row": 0
                                            }
                                        )
                                    ),
                                ),
                            ],
                        ),

                        html.Div(
                            style=CARD_STYLE,
                            children=[
                                html.H2(
                                    "Selected Candidate Info",
                                    style=SECTION_TITLE_STYLE,
                                ),
                                html.Div(
                                    id="candidate-summary",
                                    children=(
                                        build_candidate_details(
                                            initial_dataframe.iloc[
                                                0
                                            ].to_dict()
                                            if not initial_dataframe.empty
                                            else None
                                        )
                                    ),
                                ),
                            ],
                        ),
                    ],
                ),
            ],
        )
    ],
)


#############################################
# Interaction Between Components / Controller
#############################################

@app.callback(
    Output("datatable-id", "data"),
    Output("datatable-id", "page_current"),
    Output("datatable-id", "active_cell"),
    Output("datatable-id", "selected_cells"),
    Input("filter-type", "value"),
)
def update_dashboard(
    filter_type: str,
):
    """
    Update the candidate table when the rescue mission changes.
    """

    table_dataframe = get_table_dataframe(
        dashboard_results,
        filter_type,
    )

    active_cell = None

    if not table_dataframe.empty:
        active_cell = {
            "row": 0,
            "column": 0,
            "column_id": "rescue_rank",
        }

    return (
        table_dataframe.to_dict(
            "records"
        ),
        0,
        active_cell,
        [],
    )


@app.callback(
    Output(
        "datatable-id",
        "style_data_conditional",
    ),
    Output(
        "candidate-summary",
        "children",
    ),
    Output(
        "map-id",
        "children",
    ),
    Input(
        "datatable-id",
        "derived_virtual_data",
    ),
    Input(
        "datatable-id",
        "active_cell",
    ),
)
def update_selected_candidate(
    view_data,
    active_cell,
):
    """
    Highlight the clicked row and update its details and map.
    """

    candidate = get_selected_candidate(
        view_data,
        active_cell,
    )

    styles = build_selected_row_styles(
        active_cell
    )

    summary = build_candidate_details(
        candidate
    )

    if candidate is None:
        location = (
            NO_LOCATION_MESSAGE
        )
    else:
        location = candidate.get(
            "found_location",
            NO_LOCATION_MESSAGE,
        )

    map_component = build_map(
        location
    )

    return (
        styles,
        summary,
        map_component,
    )


@app.callback(
    Output("download-csv", "data"),
    Input("btn-export", "n_clicks"),
    State(
        "datatable-id",
        "derived_virtual_data",
    ),
    prevent_initial_call=True,
)
def export_table_csv(
    n_clicks,
    view_data,
):
    """
    Export the current filtered and sorted candidate view.
    """

    export_dataframe = (
        prepare_export_dataframe(
            view_data
        )
    )

    if export_dataframe.empty:
        return no_update

    return dcc.send_data_frame(
        export_dataframe.to_csv,
        "grazioso_rescue_candidates.csv",
        index=False,
    )


@app.callback(
    Output("filter-type", "value"),
    Output("datatable-id", "filter_query"),
    Output("datatable-id", "sort_by"),
    Input("btn-reset", "n_clicks"),
    prevent_initial_call=True,
)
def reset_dashboard(
    n_clicks,
):
    """
    Restore the All view and clear table filters and sorting.
    """

    return (
        "all",
        "",
        [],
    )


# Dash includes native Jupyter notebook support.
app.run(
    jupyter_mode="external",
    debug=False,
    port=8050,
)